# Cardinality-constrained structure learning for extremal graphical models

Demo on the upper Danube data: the path of incumbents, the BIC, the certified search at the selected size, the general model and two competitors. Run the cells in order; the first one clones the repository and installs the dependencies.

In [ ]:
import os
if not os.path.exists('hrl0.py'):
    !git clone -q https://github.com/manueleleonelli/certified-extremal-graphs.git
    %cd certified-extremal-graphs

In [ ]:
!pip -q install networkx scikit-learn cvxpy clarabel
import numpy as np, time
from hrl0 import *

## Data and empirical variogram

In [ ]:
X = np.load('danube_clustered.npy')
Y = data2mpareto(X, p=0.9)
Gb = emp_vario(Y)
n, d = Y.shape
flow = np.load('danube_flow_edges.npy')
flow_edges = sorted((int(min(a, b)) - 1, int(max(a, b)) - 1) for a, b in flow)
print('exceedances', n, 'stations', d)

## Path of incumbents and BIC

In [ ]:
t = time.time()
path = forward_path(Gb, kmax=60)
print('path in %.1f s' % (time.time() - t))
for k, f in sorted(path.items()):
    print(k, round(f.F, 4), round(bic(f.F, k, n), 1), round(f1_score(flow_edges, f.edges)[2], 3))
k_bic = min(path, key=lambda k: bic(path[k].F, k, n))
print('BIC size', k_bic)

## Certified search at the selected size (radius 1 takes a second, radius 2 about a minute and a half)

In [ ]:
cert = SwapCertifier(Gb, path[k_bic].edges, verbose=True)
out = cert.certify(r_max=1, time_limit=600)
print('value', out['F'], 'radius', out['radius'], 'F1 against the flow graph', f1_score(flow_edges, out['edges'])[2])

In [ ]:
out = cert.certify(r_max=2, time_limit=1800)   # certified over all 2-swaps (about 90 s)
print('value', out['F'], 'radius', out['radius'], 'refits', out['stats']['refits'])

## The general Hüsler–Reiss model (no sign constraint)

The same path and certificate without the EMTP$_2$ constraint (`signed=True`). On these data the BIC of the general model keeps decreasing up to 118 edges, so the path is shown on its sparse range; the general fit of the certified EMTP$_2$ graph has no negative conductance.

In [ ]:
path_g = forward_path(Gb, kmax=60, signed=True)
for k, f in sorted(path_g.items()):
    if k % 5 == 0: print(k, round(f.F, 4), round(bic(f.F, k, n), 1), round(f1_score(flow_edges, f.edges)[2], 3))
fit_g = fit_support(Gb, out['edges'], signed=True)
print('general fit of the certified EMTP2 graph: value', round(fit_g.F, 4), 'smallest conductance', round(fit_g.w.min(), 4))
cert_g = SwapCertifier(Gb, path_g[k_bic].edges, signed=True)
out_g = cert_g.certify(r_max=1, time_limit=600)
print('general model at k =', k_bic, ': value', out_g['F'], 'F1', round(f1_score(flow_edges, out_g['edges'])[2], 3))

## Competitors at their BIC size

In [ ]:
items = []
for rho in np.geomspace(0.01, 0.7, 14):
    S = eglearn(Gb, rho)
    if len(S) >= d - 1 and is_connected(d, S):
        F, _ = fit_general(Gb, S); items.append((len(S), F, S))
best = min(items, key=lambda it: bic(it[1], it[0], n))
print('EGlearn: edges', best[0], 'F1', round(f1_score(flow_edges, best[2])[2], 3))
full = emtp2_mle(Gb); E0 = emtp2_support(full)
print('EMTP2 support: edges', len(E0), 'F1', round(f1_score(flow_edges, E0)[2], 3))
print('MST: F1', round(f1_score(flow_edges, mst_edges(Gb))[2], 3))